``get_labels_tiered`` keeps the **positive set fixed** (``targets >= Q(q_pos)``) and sweeps **stepwise-lower negative cuts** (``targets <= Q(q_neg)``), dropping the middle band each tier. This compares CPP settings against the same positives while the negatives become more extreme. Each tier returns ``(row_mask, binary_labels)``:

In [ ]:
import numpy as np
import aaanalysis as aa

targets = np.linspace(0, 1, 40)
tiers = aa.SequenceFeature.get_labels_tiered(targets, q_pos=0.8, list_q_neg=[0.8, 0.5, 0.3])
{q: (int(mask.sum()), int(y.sum())) for q, (mask, y) in tiers.items()}  # q_neg -> (n_selected, n_positive)

Like OvO, each tier drops samples, so subset ``df_parts`` with the tier's ``row_mask`` and run a ``CPP`` per tier:

In [ ]:
df_seq = aa.load_dataset(name="DOM_GSEC", n=12)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
targets = np.linspace(0, 1, len(df_parts))

for q_neg, (row_mask, binary) in aa.SequenceFeature.get_labels_tiered(targets, q_pos=0.7, list_q_neg=[0.5, 0.3]).items():
    df_feat = aa.CPP(df_parts=df_parts[row_mask]).run(labels=binary, n_filter=3)
    print(f"q_neg={q_neg}: {row_mask.sum()} samples -> {len(df_feat)} features")

**What can go wrong?** A tier that leaves no negatives (or constant targets) raises:

In [ ]:
try:
    aa.SequenceFeature.get_labels_tiered([5.0, 5.0, 5.0, 5.0], q_pos=0.8, list_q_neg=[0.3])
except ValueError as e:
    print("ValueError:", e)